In [1]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset, random_split

In [2]:
DATA_DIR = Path("../../data")

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [15]:
raw = np.loadtxt(DATA_DIR / "train.csv", delimiter=",", skiprows=1, dtype=np.float32)
labels = torch.from_numpy(raw[:, 0]).long()
images = torch.from_numpy(raw[:, 1:]).view(-1, 1, 28, 28) / 255.0  # = ToTensor()
images = (images - 0.1307) / 0.3081                                 # = Normalize

full_dataset = TensorDataset(images, labels)
train_dataset, val_dataset = random_split(
    full_dataset, [0.9, 0.1], generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

test_raw = np.loadtxt(DATA_DIR / "test.csv", delimiter=",", skiprows=1, dtype=np.float32)
test_images = (torch.from_numpy(test_raw).view(-1, 1, 28, 28) / 255.0 - 0.1307) / 0.3081
test_loader = DataLoader(TensorDataset(test_images), batch_size=64, shuffle=False)

In [16]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)  # 출력 클래스 10개 (숫자 0~9)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))  # (N, 16, 14, 14)
        x = self.pool(self.relu(self.conv2(x)))  # (N, 32, 7, 7)
        x = x.view(-1, 32 * 7 * 7)               # Flatten
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 5

In [17]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}')

# test.csv has no labels, so measure accuracy on the validation split
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Validation Accuracy: {100 * correct / total:.2f}%')

Epoch [1/5], Loss: 0.2352
Epoch [2/5], Loss: 0.0605
Epoch [3/5], Loss: 0.0432
Epoch [4/5], Loss: 0.0318
Epoch [5/5], Loss: 0.0249
Validation Accuracy: 98.40%


In [ ]:
import matplotlib.pyplot as plt

# Predict every test image
model.eval()
preds = []
with torch.no_grad():
    for (batch,) in test_loader:
        preds.append(model(batch.to(device)).argmax(1).cpu())
preds = torch.cat(preds)

# Show the first 16 test images with their predicted labels
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    img = test_images[i, 0] * 0.3081 + 0.1307  # undo Normalize for display
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Pred: {preds[i].item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

# Kaggle submission file (ImageId starts at 1)
submission = np.column_stack([np.arange(1, len(preds) + 1), preds.numpy()])
np.savetxt(DATA_DIR / 'submission.csv', submission, fmt='%d', delimiter=',',
           header='ImageId,Label', comments='')
print('Saved', DATA_DIR / 'submission.csv')